In [1]:
import string
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
data = [
    ("How can I reset my password?", "You can reset it through account settings."),
    ("Where is my order?", "Track your package with your tracking number."),
    ("Can I get a refund?", "Refunds are processed within five business days."),
    ("How do I contact support?", "You can email our team directly anytime."),
    ("My account is locked.", "Contact an agent to unlock your account."),
    ("How do I cancel subscription?", "Go to billing settings to cancel renewal."),
    ("Do you offer discounts?", "Check our promotions page for latest coupons."),
    ("Is shipping free?", "Shipping is free for orders over fifty dollars."),
    ("Can I update shipping address?", "You can update it before item is shipped."),
    ("Why was payment declined?", "Please verify card details with your bank."),
    ("How do I delete my profile?", "Submit a deletion request via settings."),
    ("Do you ship internationally?", "Yes, we ship to over fifty countries.")
]

def clean_text(text: str) -> list[str]:
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return text.strip().split()

vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
inv_vocab = {v: k for k, v in vocab.items()}

for q, a in data:
    for word in clean_text(q) + clean_text(a):
        if word not in vocab:
            idx = len(vocab)
            vocab[word] = idx
            inv_vocab[idx] = word

vocab_size = len(vocab)

max_src_len = max(len(clean_text(q)) for q, _ in data)
max_tgt_len = max(len(clean_text(a)) for _, a in data) + 2  # враховуючи <sos> та <eos>

def text_to_tensor(text: str, max_len: int, is_target: bool = False) -> torch.Tensor:
    tokens = clean_text(text)
    if is_target:
        tokens = ["<sos>"] + tokens + ["<eos>"]
    
    indices = [vocab.get(w, vocab["<unk>"]) for w in tokens]
    if len(indices) < max_len:
        indices += [vocab["<pad>"]] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return torch.tensor(indices, dtype=torch.long)

src_tensors = torch.stack([text_to_tensor(q, max_src_len) for q, _ in data])
tgt_tensors = torch.stack([text_to_tensor(a, max_tgt_len, is_target=True) for _, a in data])

print(f"Розмір словника: {vocab_size}")
print(f"Макс. довжина запиту: {max_src_len}, Макс. довжина відповіді: {max_tgt_len}")

Розмір словника: 91
Макс. довжина запиту: 6, Макс. довжина відповіді: 10


In [ ]:
class Seq2SeqRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.encoder = nn.RNN(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.decoder = nn.RNN(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        embedded_src = self.embedding(src)
        _, hidden = self.encoder(embedded_src)
        
        embedded_tgt = self.embedding(tgt)
        outputs, _ = self.decoder(embedded_tgt, hidden)
        logits = self.fc_out(outputs)
        return logits

def train_model(model: nn.Module, src: torch.Tensor, tgt: torch.Tensor, epochs: int = 10, lr: float = 0.001):
    criterion = nn.CrossEntropyLoss(ignore_index=vocab["<pad>"])
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        decoder_input = tgt[:, :-1]
        targets = tgt[:, 1:]
        
        outputs = model(src, decoder_input)
        loss = criterion(outputs.reshape(-1, outputs.shape[-1]), targets.reshape(-1))
        loss.backward()
        optimizer.step()
        
        if epoch % 2 == 0 or epoch == epochs:
            print(f"Epoch [{epoch}/{epochs}] - Loss: {loss.item():.4f}")

print("=== Навчання RNN ===")
rnn_model = Seq2SeqRNN(vocab_size=vocab_size, hidden_size=128)
train_model(rnn_model, src_tensors, tgt_tensors, epochs=10, lr=0.001)

=== Навчання RNN ===
Epoch [2/10] - Loss: 4.4647
Epoch [4/10] - Loss: 4.1665
Epoch [6/10] - Loss: 3.8805
Epoch [8/10] - Loss: 3.6061
Epoch [10/10] - Loss: 3.3407


In [ ]:
class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.encoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        embedded_src = self.embedding(src)
        _, (hidden, cell) = self.encoder(embedded_src)
        
        embedded_tgt = self.embedding(tgt)
        outputs, _ = self.decoder(embedded_tgt, (hidden, cell))
        return self.fc_out(outputs)

print("\n=== Навчання LSTM ===")
lstm_model = Seq2SeqLSTM(vocab_size=vocab_size, hidden_size=128)
train_model(lstm_model, src_tensors, tgt_tensors, epochs=10, lr=0.001)

def generate_response(model: nn.Module, query: str, is_lstm: bool = False) -> str:
    model.eval()
    with torch.no_grad():
        src = text_to_tensor(query, max_src_len).unsqueeze(0)
        emb = model.embedding(src)
        
        if is_lstm:
            _, state = model.encoder(emb)
        else:
            _, state = model.encoder(emb)
            
        curr_token = torch.tensor([[vocab["<sos>"]]], dtype=torch.long)
        result = []
        
        for _ in range(max_tgt_len):
            out, state = model.decoder(model.embedding(curr_token), state)
            next_token = model.fc_out(out).argmax(dim=-1).item()
            if next_token == vocab["<eos>"] or next_token == vocab["<pad>"]:
                break
            result.append(inv_vocab.get(next_token, ""))
            curr_token = torch.tensor([[next_token]], dtype=torch.long)
            
    return " ".join(result)

test_queries = [
    "How can I reset my password?",
    "Where is my order?",
    "Can I get a refund?",
    "Is shipping free?",
    "Do you offer discounts?"
]

print("\n=== Порівняння генерації (5 запитів) ===")
for q in test_queries:
    rnn_out = generate_response(rnn_model, q, is_lstm=False)
    lstm_out = generate_response(lstm_model, q, is_lstm=True)
    print(f"\nЗапит: {q}")
    print(f"RNN : {rnn_out if rnn_out else '[Порожня відповідь]'}")
    print(f"LSTM: {lstm_out if lstm_out else '[Порожня відповідь]'}")


=== Навчання LSTM ===
Epoch [2/10] - Loss: 4.4677
Epoch [4/10] - Loss: 4.3676
Epoch [6/10] - Loss: 4.2652
Epoch [8/10] - Loss: 4.1574
Epoch [10/10] - Loss: 4.0409

=== Порівняння генерації (5 запитів) ===

Запит: How can I reset my password?
RNN : you can update it before item is shipped
LSTM: you can you can reset it account settings

Запит: Where is my order?
RNN : you can update it before item is shipped
LSTM: you can reset it before item shipped

Запит: Can I get a refund?
RNN : refunds are processed within five business days
LSTM: you can update it before item shipped

Запит: Is shipping free?
RNN : you can update it before item is shipped
LSTM: you can reset it account settings

Запит: Do you offer discounts?
RNN : you can update it before item is shipped
LSTM: you can reset it account settings


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
transformer_model = AutoModelForCausalLM.from_pretrained(model_name)

test_prompts = [
    "Customer: Just answer yes or no, are you a human? Support:"
]

print("=== Генерація за допомогою DistilGPT-2 ===")
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = transformer_model.generate(
        **inputs,
        max_new_tokens=125,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{response}")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 221.92it/s]


=== Генерація за допомогою DistilGPT-2 ===

Customer: Just answer yes or no, are you a human? Support: Share with others, support the people on twitter, or email you to support them. Please don't email us about our programs by email or fax. We are a full and open and open team, and are dedicated to finding the best way to get the best services for your kids.
